# 📄 Tiền xử lý dữ liệu MC-OCR

Notebook này thực hiện toàn bộ pipeline **tải xuống** và **tiền xử lý** dataset MC-OCR (Mobile-Captured OCR) để chuẩn bị cho bài toán Key Information Extraction.

## Pipeline tổng quan

1. **Setup môi trường** – Cài đặt thư viện, clone repo
2. **Tải dataset** – Download từ Google Drive, giải nén
3. **Parse CSV** – Đọc file annotation gốc
4. **EDA** – Khám phá dữ liệu (phân phối label, chất lượng ảnh, ...)
5. **Làm sạch dữ liệu** – Chuẩn hóa text, label, căn chỉnh modalities
6. **Chuyển đổi BIO** – Gán nhãn BIO cho từng token
7. **Chia tập train/val** – Stratified split
8. **Lưu kết quả** – Xuất JSONL

---
## 1. Setup môi trường

In [ ]:
# Cài đặt thư viện cần thiết
!pip install -q gdown pandas Pillow scikit-learn matplotlib

In [ ]:
import os
import re
import ast
import json
import zipfile
import logging
from copy import deepcopy
from pathlib import Path
from collections import Counter
from typing import Any, Dict, List, Tuple

import gdown
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

print('✅ Import thành công!')

In [ ]:
# ── Đường dẫn ────────────────────────────────────────────────
# Nếu chạy trên Colab, PROJECT_ROOT = /content/document-ai-vn
# Nếu chạy local, PROJECT_ROOT = thư mục gốc của project

IS_COLAB = 'COLAB_GPU' in os.environ or 'google.colab' in str(os.environ.get('PATH', ''))

if IS_COLAB:
    PROJECT_ROOT = Path('/content/document-ai-vn')
else:
    # Chạy local: notebook nằm trong notebooks/, lùi 1 bậc
    PROJECT_ROOT = Path('.').resolve().parent

DATA_ROOT = PROJECT_ROOT / 'data' / 'mcocr'
IMAGE_DIR = DATA_ROOT / 'train_images'
CSV_PATH = DATA_ROOT / 'train_df.csv'

OUTPUT_ROOT = PROJECT_ROOT / 'outputs'
PROCESSED_DIR = OUTPUT_ROOT / 'processed'

# Tạo thư mục
for path in [DATA_ROOT, IMAGE_DIR, PROCESSED_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(f'📁 PROJECT_ROOT : {PROJECT_ROOT}')
print(f'📁 DATA_ROOT    : {DATA_ROOT}')
print(f'📁 IMAGE_DIR    : {IMAGE_DIR}')
print(f'📁 CSV_PATH     : {CSV_PATH}')
print(f'📁 PROCESSED_DIR: {PROCESSED_DIR}')

---
## 2. Tải dataset MC-OCR từ Google Drive

In [ ]:
# ⚠️ THAY THẾ ID FILE GOOGLE DRIVE CỦA BẠN VÀO ĐÂY
# Cách lấy ID: mở file trên Drive → Share → Copy link
# Link dạng: https://drive.google.com/file/d/<FILE_ID>/view
# Dán <FILE_ID> vào biến bên dưới.

GDRIVE_FILE_ID = 'REPLACE_WITH_YOUR_GDRIVE_FILE_ID'

MIN_IMAGE_COUNT = 100  # Ngưỡng tối thiểu để coi dữ liệu là đã tồn tại

In [ ]:
def is_data_ready() -> bool:
    """Kiểm tra xem dữ liệu đã được giải nén đầy đủ chưa."""
    if not IMAGE_DIR.exists():
        return False
    image_count = len(list(IMAGE_DIR.glob('*.jpg')))
    return image_count >= MIN_IMAGE_COUNT


def download_and_extract() -> None:
    """Tải file ZIP từ Google Drive và giải nén vào DATA_ROOT."""
    print(f'🔍 Kiểm tra dữ liệu tại: {DATA_ROOT}')

    if is_data_ready():
        image_count = len(list(IMAGE_DIR.glob('*.jpg')))
        print(f'✅ Dữ liệu đã sẵn sàng ({image_count} ảnh). Bỏ qua bước tải.')
        return

    if GDRIVE_FILE_ID == 'REPLACE_WITH_YOUR_GDRIVE_FILE_ID':
        raise ValueError(
            '❌ Chưa cấu hình GDRIVE_FILE_ID.\n'
            'Hãy thay giá trị GDRIVE_FILE_ID ở cell phía trên bằng ID thực từ Google Drive.'
        )

    zip_path = DATA_ROOT / 'mcocr_dataset.zip'
    url = f'https://drive.google.com/uc?id={GDRIVE_FILE_ID}'

    print('⏳ Đang tải dataset từ Google Drive...')
    gdown.download(url, str(zip_path), quiet=False)

    if not zip_path.exists():
        raise FileNotFoundError(f'Tải xuống thất bại. Không tìm thấy: {zip_path}')

    print(f'📦 Đang giải nén vào {DATA_ROOT}...')
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(str(DATA_ROOT))

    os.remove(zip_path)
    print('✅ Hoàn tất tải và giải nén dataset.')


download_and_extract()

---
## 3. Parse CSV — Đọc file annotation gốc

In [ ]:
def safe_split(value: Any) -> List[str]:
    """Tách chuỗi theo delimiter '|||', bỏ qua NaN."""
    if pd.isna(value):
        return []
    return [item.strip() for item in str(value).split('|||') if str(item).strip()]


def safe_literal_eval(value: Any):
    """Parse chuỗi Python literal thành object, trả [] nếu lỗi."""
    if pd.isna(value):
        return []
    try:
        return ast.literal_eval(str(value))
    except Exception:
        return []


def parse_mcocr_csv(csv_path: str) -> List[Dict[str, Any]]:
    """Parse file CSV annotation MC-OCR thành list of dict.

    Mỗi dict chứa: image_id, texts, labels, polygons, anno_num, image_quality.
    """
    df = pd.read_csv(csv_path)
    samples: List[Dict[str, Any]] = []

    for _, row in df.iterrows():
        sample = {
            'image_id': row.get('img_id', ''),
            'texts': safe_split(row.get('anno_texts', '')),
            'labels': safe_split(row.get('anno_labels', '')),
            'polygons': safe_literal_eval(row.get('anno_polygons', '')),
            'anno_num': int(row.get('anno_num', 0)) if not pd.isna(row.get('anno_num', 0)) else 0,
            'image_quality': row.get('anno_image_quality', None),
        }
        samples.append(sample)

    return samples

In [ ]:
raw_samples = parse_mcocr_csv(str(CSV_PATH))
print(f'📊 Số mẫu raw sau parse: {len(raw_samples)}')
print()
print('--- Ví dụ 1 mẫu ---')
for key, value in raw_samples[0].items():
    print(f'  {key}: {value}')

---
## 4. EDA — Khám phá dữ liệu

In [ ]:
# 4.1 Thống kê tổng quan
total_images = len(raw_samples)
total_annotations = sum(s['anno_num'] for s in raw_samples)
avg_annotations = total_annotations / total_images if total_images > 0 else 0

print(f'🖼️  Tổng số ảnh          : {total_images:,}')
print(f'📝 Tổng số annotation    : {total_annotations:,}')
print(f'📐 Trung bình annot/ảnh  : {avg_annotations:.1f}')

In [ ]:
# 4.2 Phân phối nhãn (label distribution)
all_labels = []
for sample in raw_samples:
    all_labels.extend(sample['labels'])

label_counts = Counter(all_labels)
print('📈 Phân phối nhãn:')
for label, count in label_counts.most_common():
    pct = count / len(all_labels) * 100
    print(f'  {label:20s}: {count:6d}  ({pct:.1f}%)')

# Biểu đồ
fig, ax = plt.subplots(figsize=(10, 5))
labels_sorted = label_counts.most_common()
ax.barh([l[0] for l in labels_sorted], [l[1] for l in labels_sorted], color='#4F8EF7')
ax.set_xlabel('Số lượng')
ax.set_title('Phân phối nhãn trong MC-OCR')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# 4.3 Phân phối chất lượng ảnh
qualities = [s['image_quality'] for s in raw_samples if s['image_quality'] is not None]
qualities_float = [float(q) for q in qualities if not pd.isna(q)]

if qualities_float:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(qualities_float, bins=30, color='#FF6B6B', edgecolor='white', alpha=0.85)
    ax.set_xlabel('Chất lượng ảnh')
    ax.set_ylabel('Số lượng')
    ax.set_title('Phân phối chất lượng ảnh (image_quality)')
    plt.tight_layout()
    plt.show()
    
    print(f'\n📊 Thống kê chất lượng ảnh:')
    print(f'  Min : {min(qualities_float):.3f}')
    print(f'  Max : {max(qualities_float):.3f}')
    print(f'  Mean: {sum(qualities_float)/len(qualities_float):.3f}')
else:
    print('⚠️ Không có dữ liệu chất lượng ảnh.')

In [ ]:
# 4.4 Phân phối số annotation trên mỗi ảnh
anno_counts = [s['anno_num'] for s in raw_samples]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(anno_counts, bins=range(0, max(anno_counts) + 2), color='#51CF66', edgecolor='white', alpha=0.85)
ax.set_xlabel('Số annotation / ảnh')
ax.set_ylabel('Số lượng ảnh')
ax.set_title('Phân phối số annotation trên mỗi ảnh')
plt.tight_layout()
plt.show()

In [ ]:
# 4.5 Hiển thị ví dụ ảnh trong dataset
sample_images = list(IMAGE_DIR.glob('*.jpg'))[:6]

if sample_images:
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    for ax, img_path in zip(axes.flatten(), sample_images):
        img = Image.open(img_path)
        ax.imshow(img)
        ax.set_title(img_path.name, fontsize=9)
        ax.axis('off')
    plt.suptitle('Ví dụ ảnh trong dataset MC-OCR', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('⚠️ Không tìm thấy ảnh trong IMAGE_DIR.')

---
## 5. Làm sạch dữ liệu (Cleaning)

In [ ]:
# ── Cấu hình label ───────────────────────────────────────────
# Các entity label hợp lệ trong MC-OCR
VALID_ENTITY_LABELS = {
    'SELLER', 'ADDRESS', 'TIMESTAMP', 'TOTAL_COST', 'OTHER',
}

# Bảng chuẩn hóa: map các biến thể label về dạng chuẩn
LABEL_NORMALIZATION_MAP = {
    'SELLER': 'SELLER',
    'STORE': 'SELLER',
    'ADDRESS': 'ADDRESS',
    'ADDR': 'ADDRESS',
    'TIMESTAMP': 'TIMESTAMP',
    'DATE': 'TIMESTAMP',
    'TIME': 'TIMESTAMP',
    'TOTAL_COST': 'TOTAL_COST',
    'TOTAL': 'TOTAL_COST',
    'COST': 'TOTAL_COST',
    'OTHER': 'OTHER',
    'OTHERS': 'OTHER',
    'NONE': 'OTHER',
}

print(f'✅ Nhãn hợp lệ: {VALID_ENTITY_LABELS}')

In [ ]:
SPACE_PATTERN = re.compile(r'\s+')


def clean_text(text: str) -> str:
    """Chuẩn hóa khoảng trắng và strip text."""
    text = str(text).strip()
    text = SPACE_PATTERN.sub(' ', text)
    return text


def normalize_label(label: str) -> str:
    """Chuẩn hóa label về dạng chuẩn (uppercase + map biến thể)."""
    label = str(label).strip().upper()
    return LABEL_NORMALIZATION_MAP.get(label, label)


def align_modalities(sample: Dict) -> Dict:
    """Căn chỉnh độ dài texts, labels, polygons về cùng kích thước."""
    sample = deepcopy(sample)
    texts = sample.get('texts', [])
    labels = sample.get('labels', [])
    polygons = sample.get('polygons', [])

    target_len = min(len(texts), len(labels))
    if polygons:
        target_len = min(target_len, len(polygons))

    sample['texts'] = texts[:target_len]
    sample['labels'] = labels[:target_len]
    sample['polygons'] = polygons[:target_len] if polygons else []
    return sample


def clean_sample(sample: Dict) -> Dict:
    """Làm sạch 1 mẫu: align → clean text → normalize label → lọc invalid."""
    sample = align_modalities(sample)

    cleaned_texts: List[str] = []
    cleaned_labels: List[str] = []
    cleaned_polygons: List = []

    polygons = sample.get('polygons', [])
    for idx, (text, label) in enumerate(
        zip(sample.get('texts', []), sample.get('labels', []))
    ):
        text = clean_text(text)
        label = normalize_label(label)

        if not text:
            continue
        if label not in VALID_ENTITY_LABELS:
            continue

        cleaned_texts.append(text)
        cleaned_labels.append(label)
        if polygons:
            cleaned_polygons.append(polygons[idx])

    sample['texts'] = cleaned_texts
    sample['labels'] = cleaned_labels
    sample['polygons'] = cleaned_polygons
    return sample


print('✅ Định nghĩa hàm cleaning thành công.')

In [ ]:
# Áp dụng cleaning
cleaned_samples = [clean_sample(s) for s in raw_samples]

# Loại bỏ mẫu không còn text nào sau khi clean
cleaned_samples = [s for s in cleaned_samples if s.get('texts')]

print(f'📊 Trước cleaning : {len(raw_samples):,} mẫu')
print(f'📊 Sau cleaning   : {len(cleaned_samples):,} mẫu')
print(f'📊 Bị loại        : {len(raw_samples) - len(cleaned_samples):,} mẫu')
print()

# So sánh phân phối label trước/sau cleaning
clean_labels = []
for sample in cleaned_samples:
    clean_labels.extend(sample['labels'])

clean_label_counts = Counter(clean_labels)
print('📈 Phân phối nhãn sau cleaning:')
for label, count in clean_label_counts.most_common():
    pct = count / len(clean_labels) * 100
    print(f'  {label:20s}: {count:6d}  ({pct:.1f}%)')

---
## 6. Chuyển đổi sang định dạng BIO

In [ ]:
def polygon_to_bbox(polygon: List) -> List[int]:
    """Chuyển polygon (list tọa độ) thành bounding box [x_min, y_min, x_max, y_max]."""
    if not polygon:
        return [0, 0, 0, 0]

    # Polygon có thể là list phẳng [x1,y1,x2,y2,...] hoặc list of pairs [[x1,y1],...]
    if isinstance(polygon[0], (list, tuple)):
        x_coords = [p[0] for p in polygon]
        y_coords = [p[1] for p in polygon]
    else:
        x_coords = [polygon[i] for i in range(0, len(polygon), 2)]
        y_coords = [polygon[i] for i in range(1, len(polygon), 2)]

    return [int(min(x_coords)), int(min(y_coords)),
            int(max(x_coords)), int(max(y_coords))]


def normalize_bbox(bbox: List[int], width: int, height: int) -> List[int]:
    """Normalize bbox về khoảng [0, 1000] (chuẩn LayoutLM)."""
    if width <= 0 or height <= 0:
        return [0, 0, 0, 0]
    return [
        int(1000 * bbox[0] / width),
        int(1000 * bbox[1] / height),
        int(1000 * bbox[2] / width),
        int(1000 * bbox[3] / height),
    ]


print('✅ Định nghĩa hàm bbox thành công.')

In [ ]:
def convert_sample_to_bio(sample: Dict) -> Dict:
    """Chuyển 1 mẫu cleaned thành định dạng JSONL với BIO labels.

    Returns:
        Dict chứa image_path, image_id, words, boxes, labels.
        None nếu ảnh không tồn tại.
    """
    image_path = IMAGE_DIR / sample['image_id']
    if not image_path.exists():
        return None

    with Image.open(image_path) as img:
        width, height = img.size

    words = sample.get('texts', [])
    entities = sample.get('labels', [])
    polygons = sample.get('polygons', [])
    boxes = []
    bio_labels = []

    prev_entity = None
    for idx, entity in enumerate(entities):
        # Tính bbox
        if polygons:
            bbox = polygon_to_bbox(polygons[idx])
            bbox = normalize_bbox(bbox, width, height)
        else:
            bbox = [0, 0, 0, 0]
        boxes.append(bbox)

        # Gán nhãn BIO
        # Entity 'OTHER' là background → label 'O'
        if entity == 'OTHER':
            bio_labels.append('O')
            prev_entity = None
        else:
            prefix = 'B' if entity != prev_entity else 'I'
            bio_labels.append(f'{prefix}-{entity}')
            prev_entity = entity

    return {
        'image_path': str(image_path),
        'image_id': sample['image_id'],
        'words': words,
        'boxes': boxes,
        'labels': bio_labels,
        'image_quality': sample.get('image_quality', None),
    }


print('✅ Định nghĩa hàm BIO conversion thành công.')

In [ ]:
# Áp dụng chuyển đổi BIO
converted_samples = [convert_sample_to_bio(s) for s in cleaned_samples]
converted_samples = [s for s in converted_samples if s is not None]

print(f'📊 Sau BIO conversion: {len(converted_samples):,} mẫu hợp lệ')
print(f'📊 Bị loại (ảnh missing): {len(cleaned_samples) - len(converted_samples):,} mẫu')
print()

# Ví dụ kết quả
if converted_samples:
    example = converted_samples[0]
    print('--- Ví dụ 1 mẫu sau chuyển đổi ---')
    print(f'  image_id: {example["image_id"]}')
    print(f'  words   : {example["words"][:5]}...')
    print(f'  boxes   : {example["boxes"][:5]}...')
    print(f'  labels  : {example["labels"][:5]}...')

In [ ]:
# Phân phối nhãn BIO
bio_labels_all = []
for sample in converted_samples:
    bio_labels_all.extend(sample['labels'])

bio_counts = Counter(bio_labels_all)
print('📈 Phân phối nhãn BIO:')
for label, count in bio_counts.most_common():
    pct = count / len(bio_labels_all) * 100
    print(f'  {label:20s}: {count:6d}  ({pct:.1f}%)')

# Biểu đồ
fig, ax = plt.subplots(figsize=(10, 5))
bio_sorted = bio_counts.most_common()
colors = ['#FF6B6B' if l[0] == 'O' else '#4ECDC4' if l[0].startswith('B-') else '#45B7D1'
          for l in bio_sorted]
ax.barh([l[0] for l in bio_sorted], [l[1] for l in bio_sorted], color=colors)
ax.set_xlabel('Số lượng')
ax.set_title('Phân phối nhãn BIO')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

---
## 7. Chia tập Train / Validation

In [ ]:
VAL_SIZE = 0.2
RANDOM_STATE = 42


def build_stratify_key(sample: Dict) -> str:
    """Tạo key phân tầng dựa trên tổ hợp label + quality."""
    labels = sorted(set(sample.get('labels', [])))
    quality = sample.get('image_quality', None)

    if quality is None:
        quality_bin = 'unknown'
    else:
        try:
            q_val = float(quality)
            if q_val < 0.33:
                quality_bin = 'low'
            elif q_val < 0.66:
                quality_bin = 'medium'
            else:
                quality_bin = 'high'
        except Exception:
            quality_bin = 'unknown'

    field_key = '+'.join(labels) if labels else 'no_label'
    return f'{field_key}__{quality_bin}'


def safe_train_val_split(
    samples: List[Dict],
    val_size: float = VAL_SIZE,
    random_state: int = RANDOM_STATE,
) -> Tuple[List[Dict], List[Dict]]:
    """Chia tập train/val, cố gắng stratify theo label + quality."""
    stratify_labels = [build_stratify_key(s) for s in samples]
    counts = Counter(stratify_labels)

    # Nếu có nhóm quá nhỏ (< 2 mẫu), bỏ stratify để tránh lỗi
    if any(v < 2 for v in counts.values()):
        train_samples, val_samples = train_test_split(
            samples, test_size=val_size, random_state=random_state
        )
    else:
        train_samples, val_samples = train_test_split(
            samples,
            test_size=val_size,
            random_state=random_state,
            stratify=stratify_labels,
        )
    return train_samples, val_samples


print('✅ Định nghĩa hàm split thành công.')

In [ ]:
train_samples, val_samples = safe_train_val_split(converted_samples)

print(f'📊 Tổng mẫu   : {len(converted_samples):,}')
print(f'📊 Train       : {len(train_samples):,} ({len(train_samples)/len(converted_samples)*100:.1f}%)')
print(f'📊 Validation  : {len(val_samples):,} ({len(val_samples)/len(converted_samples)*100:.1f}%)')

---
## 8. Lưu kết quả (JSONL)

In [ ]:
def save_jsonl(data: List[Dict], output_path: Path) -> None:
    """Lưu list of dict thành file JSONL (mỗi dòng = 1 JSON object)."""
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, 'w', encoding='utf-8') as file:
        for item in data:
            file.write(json.dumps(item, ensure_ascii=False) + '\n')

    print(f'💾 Đã lưu {len(data):,} mẫu → {output_path}')


# Lưu tất cả
save_jsonl(converted_samples, PROCESSED_DIR / 'all.jsonl')
save_jsonl(train_samples, PROCESSED_DIR / 'train.jsonl')
save_jsonl(val_samples, PROCESSED_DIR / 'val.jsonl')

print()
print('✅ Pipeline tiền xử lý hoàn tất!')
print(f'📁 Kết quả tại: {PROCESSED_DIR}')

---
## 9. Kiểm tra kết quả

In [ ]:
# Đọc lại file JSONL để kiểm tra
print('📂 Danh sách file output:')
for file_path in sorted(PROCESSED_DIR.glob('*.jsonl')):
    line_count = sum(1 for _ in open(file_path, encoding='utf-8'))
    size_kb = file_path.stat().st_size / 1024
    print(f'  {file_path.name:15s} — {line_count:,} dòng — {size_kb:.1f} KB')

print()
# Đọc thử 1 mẫu từ train.jsonl
train_path = PROCESSED_DIR / 'train.jsonl'
with open(train_path, 'r', encoding='utf-8') as file:
    first_line = json.loads(file.readline())

print('--- Mẫu đầu tiên trong train.jsonl ---')
for key, value in first_line.items():
    if isinstance(value, list) and len(value) > 5:
        print(f'  {key}: {value[:5]}... (tổng {len(value)} phần tử)')
    else:
        print(f'  {key}: {value}')

In [ ]:
# Tóm tắt toàn bộ pipeline
print('=' * 60)
print('📋 TÓM TẮT PIPELINE TIỀN XỬ LÝ MC-OCR')
print('=' * 60)
print(f'  Dữ liệu raw       : {len(raw_samples):,} mẫu')
print(f'  Sau cleaning       : {len(cleaned_samples):,} mẫu')
print(f'  Sau BIO conversion : {len(converted_samples):,} mẫu')
print(f'  Train set          : {len(train_samples):,} mẫu')
print(f'  Validation set     : {len(val_samples):,} mẫu')
print(f'  Tỷ lệ train/val   : {VAL_SIZE*100:.0f}%')
print(f'  Output directory   : {PROCESSED_DIR}')
print(f'  Files              : all.jsonl, train.jsonl, val.jsonl')
print('=' * 60)